In [1]:
import sys
from pathlib import Path

# Add the project root containing 'src' to Python's search path
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src").is_dir():
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break

from src.audio.components.data_ingestion import DataIngestion
from src.audio.components.data_transformation import DataTransformation
from src.audio.components.model_training import ModelTraining
from src.audio.components.mode_pusher import ModelPusher
from src.audio.components.model_evaluation import ModelEvaluation
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import *
from src.audio.entity.artifact_entity import *

In [2]:
class TrainingPipeline:

    def __init__(self):
        self.data_ingestion_config = DataIngestionConfig()
        self.data_tranformation_config = DataTranformationConfig()
        self.model_trainer_config = ModelTrainerConfig()
        self.model_evaluation_config = ModelEvaluationConfig()

    def start_data_ingestion(self):
        try:
            data_ingestion = DataIngestion(
                data_ingestion_config=self.data_ingestion_config
            )
            data_ingestion_artifacts = data_ingestion.initiate_data_ingestion()
            return data_ingestion_artifacts
        except Exception as e:
            raise CustomException(e, sys)

    def start_data_transformation(self, data_ingestion_artifacts: DataIngestionArtifacts):
        try:
            data_transformation = DataTransformation(
                data_transformation_config=self.data_tranformation_config,
                data_ingestion_artifact=data_ingestion_artifacts
            )
            data_preprocessing_artifacts = data_transformation.initiate_data_transformation()
            return data_preprocessing_artifacts
        except Exception as e:
            raise CustomException(e, sys)

    def start_model_trainer(self, data_transformation_artifact: DataTransformationArtifacts):
        try:
            model_trainer = ModelTraining(
                data_transformation_artifact=data_transformation_artifact,
                model_trainer_config=self.model_trainer_config
            )
            model_trainer_artifact = model_trainer.initiate_model_trainer()
            return model_trainer_artifact
        except Exception as e:
            raise CustomException(e, sys)

    def start_model_evaluation(self, data_transformation_artifact: DataTransformationArtifacts,
                               model_trainer_artifacts: ModelTrainerArtifacts):
        try:
            model_evaluation = ModelEvaluation(
                self.model_evaluation_config,
                data_transformation_artifact,
                model_trainer_artifacts
            )
            model_evaluation_artifacts = model_evaluation.initiate_model_evaluation()
            return model_evaluation_artifacts
        except Exception as e:
            raise CustomException(e, sys)

    def start_model_pusher(self, model_evaluation_artifacts: ModelEvaluationArtifacts):
        try:
            required_attrs = ["is_model_accepted", "trained_model_path", "s3_model_path", "s3_model_loss"]
            missing_attrs = [attr for attr in required_attrs if not hasattr(model_evaluation_artifacts, attr)]
            if missing_attrs:
                raise ValueError(f"Required attributes are missing {missing_attrs}")

            model_pusher = ModelPusher(model_evaluation_artifact=model_evaluation_artifacts)
            model_pusher_artifacts = model_pusher.initiate_model_pusher()
            return model_pusher_artifacts
        except Exception as e:
            raise CustomException(e, sys)

    def run_pipeline(self):
        try:
            data_ingestion_artifacts = self.start_data_ingestion()
            data_transformation_artifact = self.start_data_transformation(
                data_ingestion_artifacts=data_ingestion_artifacts
            )
            model_trainer_artifact = self.start_model_trainer(
                data_transformation_artifact=data_transformation_artifact
            )
            model_evaluation_artifacts = self.start_model_evaluation(
                data_transformation_artifact,
                model_trainer_artifact
            )
            model_pusher_artifact = self.start_model_pusher(
                model_evaluation_artifacts=model_evaluation_artifacts
            )
            return model_pusher_artifact
        except Exception as e:
            raise CustomException(e, sys)